In [ ]:
import os
import math
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def compute_inter(box1, box2):
    x1 = torch.max(box1[0], box2[0])
    y1 = torch.max(box1[1], box2[1])
    x2 = torch.min(box1[2], box2[2])
    y2 = torch.min(box1[3], box2[3])

    return torch.clamp(x2 - x1, min=0) * torch.clamp(y2 - y1, min=0)   

def compute_box_area(box):
    return (box[2] - box[0]) * (box[3] - box[1])

def compute_iou(box1, box2):
    inter = compute_inter(box1, box2)
    area1 = compute_box_area(box1)
    area2 = compute_box_area(box2)
    union = area1 + area2 - inter

    return inter / (union + 1e-7)

def compute_giou(box1, box2):
    iou = compute_iou(box1, box2)
    
    cx1 = torch.min(box1[0], box2[0])
    cy1 = torch.min(box1[1], box2[1])
    cx2 = torch.max(box1[2], box2[2])
    cy2 = torch.max(box1[3], box2[3])
    c_area = (cx2 - cx1) * (cy2 - cy1)

    area1 = compute_box_area(box1)
    area2 = compute_box_area(box2)

    inter = compute_inter(box1, box2)
    union = area1 + area2 - inter

    giou = iou - (c_area - union) / (c_area + 1e-7)
    return giou

def compute_ciou(box1, box2):
    iou = compute_iou(box1, box2)

    dx1 = (box1[0] + box1[2]) / 2
    dy1 = (box1[1] + box1[3]) / 2
    dx2 = (box2[0] + box2[2]) / 2
    dy2 = (box2[1] + box2[3]) / 2
    d2 = (dx1 - dx2) ** 2 + (dy1 - dy2) ** 2

    cx1 = torch.min(box1[0], box2[0])
    cy1 = torch.min(box1[1], box2[1])
    cx2 = torch.max(box1[2], box2[2])
    cy2 = torch.max(box1[3], box2[3])
    c2 = (cx2 - cx1) ** 2 + (cy2 - cy1) ** 2

    w1 = box1[2] - box1[0]
    h1 = box1[3] - box1[1]
    w2 = box2[2] - box2[0]
    h2 = box2[3] - box2[1]

    v = (4 / math.pi ** 2) * (torch.atan(w2 / (h2 + 1e-7)) - torch.atan(w1 / (h1 + 1e-7))) ** 2

    with torch.no_grad():
        alpha = v / (1 - iou + v + 1e-7)

    ciou = iou - d2 / (c2 + 1e-7) - alpha * v
    return ciou




# -- 시나리오 비교 --
# GT는 (100, 100)~(200, 200)의 100x100 정사각형, 중심은 (150, 150)으로 고정
gt = torch.tensor([100., 100., 200., 200.]) # 정답 박스 (x1, y1, x2, y2)

# 6가지 시나리오: (시각화용 영어 제목, pred 박스, 콘솔 출력용 한국어 설명)
# title은 matplotlib에 한글 폰트가 없을 때 깨지므로 영어, desc는 콘솔 출력용이라 한국어 유지
scenarios = [
    ("Nearly accurate",
     torch.tensor([102., 98., 202., 198.]),
     "GT와 거의 일치 -> IoU 1에 근접"),

    ("Partial overlap",
     torch.tensor([120., 110., 220., 210.]),
     "대각선으로 약간 이동 -> 일반적인 학습 중 상황"),

    ("Adjacent (IoU=0)",
     torch.tensor([220., 100., 320., 200.]),
     "GT 바로 옆 (IoU=0) -> GIoU는 약한 음수로 거리 신호 전달"),

    ("Far apart (IoU=0)",
     torch.tensor([400., 400., 500., 500.]),
     "GT에서 멀리 (IoU=0) -> GIoU가 더 작음 (GIoU의 거리 민감도)"),

    ("Pred contains GT",
     torch.tensor([50., 50., 250., 250.]),
     "pred가 GT를 완전히 감쌈 -> 외접 영역 C = pred, GIoU=IoU"),

    ("Same center, diff aspect",
     torch.tensor([100., 130., 200., 170.]),
     "중심점은 같으나 가로로 납작 -> CIoU의 종횡비 패널티가 드러남"),
]

print(f"\nGT: {gt.tolist()} (100x100 정사각형, 중심 (150, 150))")
for title, pred, desc in scenarios: # 시나리오마다 IoU/GIoU/CIoU 출력
    iou = compute_iou(pred, gt).item()
    giou = compute_giou(pred, gt).item()
    ciou = compute_ciou(pred, gt).item()
    print(f"\n[{title}] {desc}")
    print(f"  Pred: {pred.tolist()}")
    print(f"  IoU={iou:.4f}, GIoU={giou:.4f}, CIoU={ciou:.4f}")


# -- 시각화 --
fig, axes = plt.subplots(2, 3, figsize=(15, 10)) # 6개 시나리오를 2x3 그리드로 배치
axes = axes.flatten() # 2x3을 1차원으로 펴서 zip으로 순회


for ax, (title, pred, _) in zip(axes, scenarios): # 시나리오마다 박스 그리기
    ax.set_xlim(0, 550)
    ax.set_ylim(550, 0) # y축 뒤집기 (이미지 좌표계는 위가 0)
    ax.set_aspect('equal') # 가로세로 비율 동일하게
    ax.set_title(title, fontsize=11)


    # GT Box (녹색)
    rect_gt = patches.Rectangle(
        (gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1], # (좌상단 좌표), 너비, 높이
        linewidth=2, edgecolor='green', facecolor='green', alpha=0.3,
        label='GT')
    ax.add_patch(rect_gt) # 그래프에 사각형 추가


    # Pred Box (빨간)
    rect_pred = patches.Rectangle(
        (pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1],
        linewidth=2, edgecolor='red', facecolor='red', alpha=0.3,
        label='Pred')
    ax.add_patch(rect_pred)


    iou_val = compute_iou(pred, gt).item()
    giou_val = compute_giou(pred, gt).item()
    ciou_val = compute_ciou(pred, gt).item()
    ax.text(275, 520, # 각 subplot 하단 중앙에 세 지표 모두 표시
            f"IoU={iou_val:.3f}  GIoU={giou_val:.3f}  CIoU={ciou_val:.3f}",
            fontsize=9, ha='center')
    ax.legend(loc='upper right', fontsize=8)


plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/iou_comparison.png', dpi=100) # 결과를 이미지 파일로 저장
print("\n시각화 저장: outputs/iou_comparison.png")



GT: [100.0, 100.0, 200.0, 200.0] (100x100 정사각형, 중심 (150, 150))

[Nearly accurate] GT와 거의 일치 -> IoU 1에 근접
  Pred: [102.0, 98.0, 202.0, 198.0]
  IoU=0.9238, GIoU=0.9230, CIoU=0.9234

[Partial overlap] 대각선으로 약간 이동 -> 일반적인 학습 중 상황
  Pred: [120.0, 110.0, 220.0, 210.0]
  IoU=0.5625, GIoU=0.5322, CIoU=0.5436

[Adjacent (IoU=0)] GT 바로 옆 (IoU=0) -> GIoU는 약한 음수로 거리 신호 전달
  Pred: [220.0, 100.0, 320.0, 200.0]
  IoU=0.0000, GIoU=-0.0909, CIoU=-0.2466

[Far apart (IoU=0)] GT에서 멀리 (IoU=0) -> GIoU가 더 작음 (GIoU의 거리 민감도)
  Pred: [400.0, 400.0, 500.0, 500.0]
  IoU=0.0000, GIoU=-0.8750, CIoU=-0.5625

[Pred contains GT] pred가 GT를 완전히 감쌈 -> 외접 영역 C = pred, GIoU=IoU
  Pred: [50.0, 50.0, 250.0, 250.0]
  IoU=0.2500, GIoU=0.2500, CIoU=0.2500

[Same center, diff aspect] 중심점은 같으나 가로로 납작 -> CIoU의 종횡비 패널티가 드러남
  Pred: [100.0, 130.0, 200.0, 170.0]
  IoU=0.4000, GIoU=0.4000, CIoU=0.3934

시각화 저장: iou_comparison.png

 실습 1 완료!


In [6]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def compute_iou_np(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    iou = inter / (area1 + area2 - inter + 1e-7)
    return iou

# -- 가상 검출 결과 --
# 5개의 GT box가 있는 이미지
gt_boxes = [ # 정답 박스 목록 [x1, y1, x2, y2]
    [50, 50, 150, 150], # 객체 0
    [200, 100, 350, 250], # 객체 1
    [400, 200, 500, 350], # 객체 2
    [100, 300, 250, 450], # 객체 3
    [350, 350, 480, 480], # 객체 4
]


# 모델의 예측 결과 (confidence 순으로 정렬)
predictions = [ # 모델 예측 목록: 박스 + confidence (신뢰도)
    {"box": [55, 48, 155, 148], "conf": 0.95}, # GT0과 매칭
    {"box": [205, 105, 345, 245], "conf": 0.90}, # GT1과 매칭
    {"box": [300, 300, 400, 400], "conf": 0.85}, # 어떤 GT와도 안 맞음 (FP)
    {"box": [60, 55, 145, 145], "conf": 0.80}, # GT0과 중복 (이미 매칭됨)
    {"box": [405, 205, 495, 345], "conf": 0.70}, # GT2와 매칭
    {"box": [105, 305, 245, 445], "conf": 0.60}, # GT3과 매칭
    {"box": [10, 10, 30, 30], "conf": 0.50}, # FP (잘못된 검출)
]


print(f"\nGT 객체 수: {len(gt_boxes)}")
print(f"예측 수: {len(predictions)}")

iou_threshold = 0.5
matched_gt = set()
precisions = []
recalls = []

tp_list = []
fp_list = []

for i, pred in enumerate(predictions):
    best_iou = 0
    best_gt_idx = -1

    for j, gt in enumerate(gt_boxes):
        iou = compute_iou_np(pred["box"], gt)
        if iou > best_iou:
            best_iou = iou
            best_gt_idx = j

    if best_iou >= iou_threshold and best_gt_idx not in matched_gt:
        tp_list.append(1)
        fp_list.append(0)
        matched_gt.add(best_gt_idx)
        status = "TP"
    else:
        tp_list.append(0)
        fp_list.append(1)
        status = "FP"

    tp_cumsum = sum(tp_list)
    fp_cumsum = sum(fp_list)
    precision = tp_cumsum / (tp_cumsum + fp_cumsum)
    recall = tp_cumsum / len(gt_boxes)

    precisions.append(precision)
    recalls.append(recall)

    print(f"예측 {i}: conf={pred['conf']:.2f}, IoU={best_iou:.3f}, "
          f"{status}, Precision={precision:.3f}, Recall={recall:.3f}")
    
# -- AP 계산 (11-point interpolation) --
recall_levels = np.linspace(0, 1, 11) # recall 기준점 11개 (0, 0.1, ..., 1.0)
ap = 0
for r_level in recall_levels:
    # r_level 이상의 recall에서 최대 precision
    prec_at_level = [p for p, r in zip(precisions, recalls) if r >= r_level]
    if prec_at_level:
        ap += max(prec_at_level) # 각 기준점의 최대 precision을 누적


ap /= 11 # 11개 기준점의 평균이 AP
print(f"\n AP@0.5 (11-point): {ap:.4f}")


# -- PR Curve 시각화 --
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.plot(recalls, precisions, 'b-o', linewidth=2, markersize=8) # recall-precision 곡선
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Precision-Recall Curve (AP@0.5 = {ap:.3f})', fontsize=14)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.fill_between(recalls, precisions, alpha=0.2) # 곡선 아래 영역 (넓을수록 AP 큼)


plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/pr_curve.png', dpi=100) # 결과 이미지 저장
print("PR Curve 저장: outputs/pr_curve.png")
    


GT 객체 수: 5
예측 수: 7
예측 0: conf=0.95, IoU=0.871, TP, Precision=1.000, Recall=0.200
예측 1: conf=0.90, IoU=0.871, TP, Precision=1.000, Recall=0.400
예측 2: conf=0.85, IoU=0.102, FP, Precision=0.667, Recall=0.400
예측 3: conf=0.80, IoU=0.765, FP, Precision=0.500, Recall=0.400
예측 4: conf=0.70, IoU=0.840, TP, Precision=0.600, Recall=0.600
예측 5: conf=0.60, IoU=0.871, TP, Precision=0.667, Recall=0.800
예측 6: conf=0.50, IoU=0.000, FP, Precision=0.571, Recall=0.800

 AP@0.5 (11-point): 0.6970
PR Curve 저장: pr_curve.png


In [3]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def nms(boxes, scores, iou_threshold=0.5):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)

    order = scores.argsort()[::-1]
    keep = []

    while order.size > 0:
        i = order[0]
        keep.append(i)

        if order.size == 1:
            break

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        iou = inter / (areas[i] + areas[order[1:]] - inter + 1e-7)
        remaining = np.where(iou < iou_threshold)[0]
        order = order[remaining + 1] 

    return keep

boxes = np.array([
    [100, 100, 210, 210],
    [105, 108, 215, 215],
    [110, 105, 220, 218],
    [300, 300, 420, 420],
    [305, 310, 425, 425],
    [500, 100, 600, 200],
], dtype=np.float32)


scores = np.array([0.9, 0.85, 0.7, 0.95, 0.6, 0.8])

print(f"\nNMS 전: {len(boxes)}개 BBox")
for i, (box, score) in enumerate(zip(boxes, scores)):
    print(f"BBox {i}: {box.tolist()}, conf={score:.2f}")

keep = nms(boxes, scores, iou_threshold=0.5)
print(f"\nNMS 후: {len(keep)}개 BBox (유지: {keep})")
for i in keep:
    print(f"BBox {i}: {boxes[i].tolist()}, conf={scores[i]:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = ["red", "blue", "green", "orange", "purple", "cyan"]

ax1.set_title("Before NMS", fontsize=14)
ax1.set_xlim(0, 700)
ax1.set_ylim(500, 0)
for i, (box, score) in enumerate(zip(boxes, scores)):
    rect = patches.Rectangle(
        (box[0], box[1]), box[2]-box[0], box[3]-box[1],
        linewidth=2, edgecolor=colors[i], facecolor=colors[i],
        alpha=0.3)
    ax1.add_patch(rect)
    ax1.text(box[0], box[1]-5, f"#{i} conf={score:.2f}",
             fontsize=9, color=colors[i])


# NMS 후
ax2.set_title("After NMS", fontsize=14)
ax2.set_xlim(0, 700)
ax2.set_ylim(500, 0)
for i in keep: 
    box = boxes[i]
    rect = patches.Rectangle(
        (box[0], box[1]), box[2]-box[0], box[3]-box[1],
        linewidth=3, edgecolor=colors[i], facecolor=colors[i],
        alpha=0.3)
    ax2.add_patch(rect)
    ax2.text(box[0], box[1]-5, f"#{i} conf={scores[i]:.2f}",
             fontsize=9, color=colors[i], fontweight='bold')


plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/nms_result.png', dpi=100)
print("\n시각화 저장: outputs/nms_result.png")


NMS 전: 6개 BBox
BBox 0: [100.0, 100.0, 210.0, 210.0], conf=0.90
BBox 1: [105.0, 108.0, 215.0, 215.0], conf=0.85
BBox 2: [110.0, 105.0, 220.0, 218.0], conf=0.70
BBox 3: [300.0, 300.0, 420.0, 420.0], conf=0.95
BBox 4: [305.0, 310.0, 425.0, 425.0], conf=0.60
BBox 5: [500.0, 100.0, 600.0, 200.0], conf=0.80

NMS 후: 3개 BBox (유지: [np.int64(3), np.int64(0), np.int64(5)])
BBox 3: [300.0, 300.0, 420.0, 420.0], conf=0.95
BBox 0: [100.0, 100.0, 210.0, 210.0], conf=0.90
BBox 5: [500.0, 100.0, 600.0, 200.0], conf=0.80

시각화 저장: nms_result.png


/tmp/ipykernel_689/3342897508.py:88: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_689/3342897508.py:88: UserWarning: Glyph 54980 (\N{HANGUL SYLLABLE HU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_689/3342897508.py:89: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from font(s) DejaVu Sans.
  plt.savefig('nms_result.png', dpi=100)
/tmp/ipykernel_689/3342897508.py:89: UserWarning: Glyph 54980 (\N{HANGUL SYLLABLE HU}) missing from font(s) DejaVu Sans.
  plt.savefig('nms_result.png', dpi=100)


In [4]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print(f"모델명: {model.model.yaml.get('yaml_file', 'yolo11n')}")
print(f"총 파라미터: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"학습 가능 파라미터: {sum(p.numel() for p in model.model.parameters() if p.requires_grad):,}")

for i, layer in enumerate(model.model.model):
    params = sum(p.numel() for p in layer.parameters())
    print(f"Layer {i:2d}: {layer.__class__.__name__:20s} -> 파라미터: {params:>8,}")

model_sizes = { 
    'yolo11n': '2.6M params, 6.5 GFLOPs, mAP 39.5',
    'yolo11s': '9.4M params, 21.5 GFLOPs, mAP 47.0',
    'yolo11m': '20.1M params, 68.0 GFLOPs, mAP 51.5',
    'yolo11l': '25.3M params, 86.9 GFLOPs, mAP 53.4',
    'yolo11x': '56.9M params, 194.9 GFLOPs, mAP 54.7',
}
for name, info in model_sizes.items():
    print(f"{name}: {info}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
모델명: yolo11n.yaml
총 파라미터: 2,624,080
학습 가능 파라미터: 0
Layer  0: Conv                 -> 파라미터:      464
Layer  1: Conv                 -> 파라미터:    4,672
Layer  2: C3k2                 -> 파라미터:    6,640
Layer  3: Conv                 -> 파라미터:   36,992
Layer  4: C3k2                 -> 파라미터:   26,080
Layer  5: Conv                 -> 파라미터:  147,712
Layer  6: C3k2                 -> 파라미터:   87,040
Layer  7: Conv                 -> 파라미터:  295,424
Layer  8: C3k2                 -> 파라미터:  346,112
Layer  9: SPPF                 -> 파라미터:  164,608
Layer 10: C2PSA                -> 파라미터:  249,728
Layer 11: Upsample             -> 파라미터:        0
Layer 12: Concat               -> 파라미터:        0
La